# Train, evaluate, and extrapolate a SpecMine baseline

This notebook uses only specs with nonblank labels from the selected CSV for training and holdout evaluation. It keeps repositories disjoint for evaluation, then refits the selected Logistic Regression or Random Forest model on all labeled specs and predicts a random sample of specs whose IDs do not appear in the CSV. Blank-label rows are treated as already reviewed and excluded from prediction. Predictions are model outputs, not verified authorship labels.

In [ ]:
import logging
import os
import sys
import tempfile
import uuid
from pathlib import Path

project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / "src" / "specmine").is_dir()),
    None,
)
if project_root is None:
    raise FileNotFoundError(
        "Could not find src/specmine. Open this notebook from the project "
        "folder or one of its subfolders."
    )
sys.path.insert(0, str(project_root / "src"))

import joblib
import numpy as np
import pandas as pd
from sklearn.base import clone

from specmine.data import load_data, load_feature_data
from specmine.evaluate import escape_spreadsheet_formula
from specmine.features import build_features
from specmine.labels import SPEC_ID_COLUMN, read_labels_csv
from specmine.train import (
    ID_COLUMN,
    REPOSITORY_COLUMN,
    prepare_training_data,
    train_models,
)

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logging.getLogger("specmine.train").setLevel(logging.INFO)

SEED = 42
TEST_SIZE = 0.2
MODEL_NAME = "logistic_regression"  # Choose "logistic_regression" or "random_forest".
PREDICTION_SAMPLE_SIZE = 100
LABELS_CSV = project_root / "examples" / "synthetic_labels.csv"
MODEL_DIR = project_root / "models"
PREDICTIONS_CSV = project_root / "notebooks" / "unlabeled_predictions.csv"

print("Labels CSV: examples/synthetic_labels.csv")
print("Prediction output base: notebooks/unlabeled_predictions.csv")

## Load labeled data

`LABELS_CSV` defaults to `examples/synthetic_labels.csv`, which contains real, manually verified labels. Change it only when using a different label set. The full metadata table is used to match IDs, but feature rows are loaded only for nonblank labeled specs at this stage. Training requires labeled examples from at least two repositories for its repository-disjoint holdout.

In [ ]:
if MODEL_NAME not in {"logistic_regression", "random_forest"}:
    raise ValueError("MODEL_NAME must be 'logistic_regression' or 'random_forest'.")
if PREDICTION_SAMPLE_SIZE < 1:
    raise ValueError("PREDICTION_SAMPLE_SIZE must be a positive integer.")

labels_frame = read_labels_csv(LABELS_CSV)
metadata = load_data(seed=SEED)
if metadata is None:
    raise RuntimeError("SpecMine metadata loading failed; training cannot continue.")

training_features, training_labels, training_repositories = prepare_training_data(
    metadata,
    labels_frame,
    seed=SEED,
)
if training_repositories.nunique() < 2:
    raise ValueError(
        "Training needs labeled specs from at least two repositories. "
        "Review another repository in notebooks/label_specs.ipynb, then "
        "add its labels to the CSV."
    )
print(f"Labeled specs used for training/evaluation: {len(training_features)}")
print(f"Labeled repositories: {training_repositories.nunique()}")
print(f"Label counts: {training_labels.value_counts().sort_index().to_dict()}")

## Evaluate on a repository-disjoint holdout

Both baselines are trained and saved to `models/` for use by `results.ipynb`; `MODEL_NAME` selects which one is refit on all labeled specs for the unlabeled predictions below. The saved `{model}.joblib` files are holdout-fitted, so they can be evaluated on the same repository-disjoint test split. The all-labeled prediction model is saved separately as `{model}_all_labeled.joblib`. Reported metrics compare predictions to the supplied labels in the holdout only; they are not authorship-accuracy estimates unless those labels are verified ground truth.

In [ ]:
training_results = train_models(
    training_features,
    training_labels,
    training_repositories,
    model_names=["logistic_regression", "random_forest"],
    test_size=TEST_SIZE,
    seed=SEED,
    model_dir=MODEL_DIR,
)
holdout_result = training_results[MODEL_NAME]
print(f"Saved holdout baselines to: {MODEL_DIR}")
print(
    f"{MODEL_NAME}: {holdout_result['n_train']} train specs from "
    f"{holdout_result['n_train_repositories']} repositories; "
    f"{holdout_result['n_test']} holdout specs from "
    f"{holdout_result['n_test_repositories']} repositories"
)
display(pd.DataFrame([holdout_result["metrics"]], index=[MODEL_NAME]))

## Refit on labeled data and predict unlabeled specs

The holdout model above is used only for evaluation. This cell clones its estimator and fits it on all nonblank labeled examples, then loads features only for a random sample of unlabeled specs. Every `spec_id` present in the CSV is excluded from that sample, including blank-label rows. The output contains predictions and positive-class probabilities, not ground-truth labels.

In [ ]:
reviewed_ids = set(labels_frame[SPEC_ID_COLUMN].astype("string"))
metadata_ids = metadata[ID_COLUMN].astype("string")
prediction_pool = metadata.loc[~metadata_ids.isin(reviewed_ids)].dropna(
    subset=[ID_COLUMN, REPOSITORY_COLUMN]
)
prediction_pool = prediction_pool.drop_duplicates(subset=[ID_COLUMN])
if prediction_pool.empty:
    raise ValueError("No specs remain outside the IDs in the labels CSV.")

candidate_order = prediction_pool.sample(
    frac=1,
    random_state=SEED,
).reset_index(drop=True)
target_count = min(PREDICTION_SAMPLE_SIZE, len(candidate_order))
selected_metadata_parts = []
selected_feature_parts = []
unavailable_count = 0
cursor = 0
selected_count = 0

while selected_count < target_count and cursor < len(candidate_order):
    needed = target_count - selected_count
    batch_size = min(max(needed, 25), len(candidate_order) - cursor)
    candidate_batch = candidate_order.iloc[cursor : cursor + batch_size]
    cursor += batch_size

    feature_batch = load_feature_data(candidate_batch[[ID_COLUMN]])
    if feature_batch is None or feature_batch.empty:
        unavailable_count += len(candidate_batch)
        continue

    feature_ids = set(feature_batch[ID_COLUMN].astype("string"))
    matched_candidates = candidate_batch.loc[
        candidate_batch[ID_COLUMN].astype("string").isin(feature_ids)
    ]
    unavailable_count += len(candidate_batch) - len(matched_candidates)
    matched_candidates = matched_candidates.iloc[:needed]
    matched_ids = set(matched_candidates[ID_COLUMN].astype("string"))
    selected_metadata_parts.append(matched_candidates)
    selected_feature_parts.append(
        feature_batch.loc[feature_batch[ID_COLUMN].astype("string").isin(matched_ids)]
    )
    selected_count += len(matched_candidates)

if selected_count == 0:
    raise ValueError("No precomputed features were found for unlabeled specs.")
if selected_count < target_count:
    raise ValueError(
        f"Only {selected_count} of {target_count} requested specs have "
        "precomputed features. Increase the available metadata pool or "
        "reduce PREDICTION_SAMPLE_SIZE."
    )
prediction_metadata = pd.concat(selected_metadata_parts, ignore_index=True)
unlabeled_feature_rows = pd.concat(selected_feature_parts, ignore_index=True)
if unavailable_count:
    print(f"Skipped {unavailable_count} specs without precomputed features.")
print(f"Random unlabeled specs selected: {len(prediction_metadata)}")

unlabeled_features = build_features(unlabeled_feature_rows)
feature_columns = holdout_result["feature_columns"]
missing_columns = [
    column for column in feature_columns if column not in unlabeled_features
]
if missing_columns:
    raise KeyError(f"Prediction features are missing model columns: {missing_columns}")

selected_ids = set(prediction_metadata[ID_COLUMN].astype("string"))
found_ids = set(unlabeled_features[ID_COLUMN].astype("string"))
missing_ids = selected_ids - found_ids
if missing_ids:
    raise ValueError(
        f"Precomputed features are missing {len(missing_ids)} selected spec(s); "
        "no incomplete prediction file will be written."
    )
prediction_rows = prediction_metadata[[ID_COLUMN, REPOSITORY_COLUMN]].merge(
    unlabeled_features,
    on=ID_COLUMN,
    how="inner",
    validate="one_to_one",
)
if len(prediction_rows) != len(prediction_metadata):
    raise RuntimeError("Prediction rows do not match the feature-backed sample size.")

training_numeric = (
    training_features[feature_columns]
    .apply(pd.to_numeric, errors="coerce")
    .replace([np.inf, -np.inf], np.nan)
    .astype(float)
)
prediction_numeric = (
    prediction_rows[feature_columns]
    .apply(pd.to_numeric, errors="coerce")
    .replace([np.inf, -np.inf], np.nan)
    .astype(float)
)
final_model = clone(holdout_result["model"])
final_model.fit(training_numeric, training_labels)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
final_model_path = MODEL_DIR / f"{MODEL_NAME}_all_labeled.joblib"
joblib.dump(final_model, final_model_path)
print(f"Saved all-labeled prediction model to: {final_model_path}")
positive_class_index = list(final_model.classes_).index(1)
probability_ai = final_model.predict_proba(prediction_numeric)[:, positive_class_index]

predictions = pd.DataFrame(
    {
        SPEC_ID_COLUMN: prediction_rows[ID_COLUMN].astype("string"),
        REPOSITORY_COLUMN: prediction_rows[REPOSITORY_COLUMN],
        "predicted_label": (probability_ai >= 0.5).astype("int8"),
        "probability_ai_assisted": probability_ai,
    }
)
predictions = predictions.apply(
    lambda column: column.map(escape_spreadsheet_formula)
)
PREDICTIONS_CSV.parent.mkdir(parents=True, exist_ok=True)
run_id = uuid.uuid4().hex
run_output = PREDICTIONS_CSV.with_name(
    f"{PREDICTIONS_CSV.stem}_{run_id}{PREDICTIONS_CSV.suffix}"
)
temporary_output = None
try:
    with tempfile.NamedTemporaryFile(
        mode="w",
        encoding="utf-8",
        newline="",
        dir=PREDICTIONS_CSV.parent,
        prefix=f".{PREDICTIONS_CSV.stem}_{run_id}_",
        suffix=".tmp",
        delete=False,
    ) as temporary_file:
        temporary_output = Path(temporary_file.name)
    predictions.to_csv(temporary_output, index=False)
    os.replace(temporary_output, run_output)
    temporary_output = None
finally:
    if temporary_output is not None:
        temporary_output.unlink(missing_ok=True)
print(f"Saved {len(predictions)} predictions to {run_output.name}")
display(predictions.head(20))